In [75]:
from sklearn.datasets import load_digits #import the dataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [76]:
digits = load_digits()
X, y = digits.data, digits.target

#split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Scaling the dataset features using Standard Scalar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [77]:
#applying one hot encoding
def one_hot_encode(y, num_classes=10):
    one_hot = np.zeros((len(y), num_classes))
    one_hot[np.arange(len(y)), y] = 1
    return one_hot

y_train_encoded = one_hot_encode(y_train)
y_test_encoded = one_hot_encode(y_test)

In [78]:
#initializing parameters for weights and biases
def initialize_parameters():
    np.random.seed(42)
    w_1 = np.random.randn(64, 30) * np.sqrt(2 / 64)
    b_1 = np.zeros((1, 30))
    w_2 = np.random.randn(30, 10) * np.sqrt(2 / 30)
    b_2 = np.zeros((1, 10))

    return w_1, b_1, w_2, b_2

In [79]:
#activation functions
def sigmoid(x):
    return 1/(1 + np.exp(-x))

def sigmoid_der(x):
    s = sigmoid(x)
    return s*(1-s)

def relu(x):
    return np.maximum(0, x)

def relu_der(x):
    return np.where(x > 0, 1, 0)

def tanh(x):
    return np.tanh(x)

def tanh_der(x):
    return 1 - np.tanh(x)**2
    

In [80]:
#Forward
def forward(X, w_1, b_1, w_2, b_2, activation): 
    a_0 = X
    z_1 = a_0 @ w_1 + b_1
    if activation == 'relu':
        a_1 = relu(z_1)
    elif activation == 'tanh':
        a_1 = tanh(z_1)
    elif activation == 'sigmoid': 
        a_1 = sigmoid(z_1)

    z_2 = a_1 @ w_2 + b_2
    
    if activation == 'relu':
        a_2 = relu(z_2)
    elif activation == 'tanh':
        a_2 = tanh(z_2)
    elif activation == 'sigmoid': 
        a_2 = sigmoid(z_2)

    return z_1, a_1, z_2, a_2
    

In [81]:
#backpropagation
def backpropagation(X, y, z_1, a_1, z_2, a_2, w_2, activation):
    m = X.shape[0]

    dz_2 = a_2 - y
    dw_2 = (1/m)*(a_1.T @ dz_2)
    db_2 = (1/m)*np.sum(dz_2, axis=0, keepdims=True)

    if activation == 'relu':
        da_1 = dz_2 @ w_2.T
        dz_1 = da_1 * relu_der(z_1)
    elif activation == 'tanh':
        da_1 = dz_2 @ w_2.T
        dz_1 = da_1 * tanh_der(z_1)
    elif activation == 'sigmoid':
        da_1 = dz_2 @ w_2.T
        dz_1 = da_1 * sigmoid_der(z_1)

    dw_1 = (1/m)*(X.T @ dz_1)
    db_1 = (1/m)*np.sum(dz_1, axis=0, keepdims=True)

    return dw_1, db_1, dw_2, db_2

In [82]:
#update params
def update_params(w_1, b_1, w_2, b_2, dw_1, db_1, dw_2, db_2, learning_rate):
    w_1 = w_1 - learning_rate*dw_1
    b_1 = b_1 - learning_rate*db_1
    w_2 = w_2 - learning_rate*dw_2
    b_2 = b_2 - learning_rate*db_2
    
    return w_1, b_1, w_2, b_2

In [83]:
def train(X_train, y_train_encoded, X_test, y_test, y_test_encoded, activation, epoch, learning_rate):
    
    w_1, b_1, w_2, b_2 = initialize_parameters()
    
    for epoch in range(100):
        z_1, a_1, z_2, a_2 = forward(X_train, w_1, b_1, w_2, b_2, activation=activation)
        dw_1, db_1, dw_2, db_2 = backpropagation(X_train, y_train_encoded, z_1, a_1, z_2, a_2, w_2, activation=activation)
        w_1, b_1, w_2, b_2 = update_params(w_1, b_1, w_2, b_2, dw_1, db_1, dw_2, db_2, learning_rate)

        if epoch % 10 == 0:
            z1_test, a1_test, z2_test, a2_test = forward(X_test, w_1, b_1, w_2, b_2, activation=activation)
            y_pred = np.argmax(a2_test, axis=1)
            acc = accuracy_score(y_test, y_pred)
            print(f"Epoch: {epoch} | Test Accuracy: {acc}")
    
    return w_1, b_1, w_2, b_2

In [84]:
sigmoid_train = train(
    X_train, y_train_encoded, 
    X_test, y_test, y_test_encoded, 
    activation='sigmoid',
    epoch = 100,
    learning_rate=0.5
)

Epoch: 0 | Test Accuracy: 0.325
Epoch: 10 | Test Accuracy: 0.7361111111111112
Epoch: 20 | Test Accuracy: 0.825
Epoch: 30 | Test Accuracy: 0.8666666666666667
Epoch: 40 | Test Accuracy: 0.8805555555555555
Epoch: 50 | Test Accuracy: 0.8972222222222223
Epoch: 60 | Test Accuracy: 0.9055555555555556
Epoch: 70 | Test Accuracy: 0.925
Epoch: 80 | Test Accuracy: 0.925
Epoch: 90 | Test Accuracy: 0.9305555555555556


In [85]:
relu_train = train(
    X_train, y_train_encoded, 
    X_test, y_test, y_test_encoded, 
    activation='relu',
    epoch = 100,
    learning_rate=0.5
)

Epoch: 0 | Test Accuracy: 0.25555555555555554
Epoch: 10 | Test Accuracy: 0.8111111111111111
Epoch: 20 | Test Accuracy: 0.8722222222222222
Epoch: 30 | Test Accuracy: 0.9055555555555556
Epoch: 40 | Test Accuracy: 0.9166666666666666
Epoch: 50 | Test Accuracy: 0.925
Epoch: 60 | Test Accuracy: 0.9305555555555556
Epoch: 70 | Test Accuracy: 0.9333333333333333
Epoch: 80 | Test Accuracy: 0.9361111111111111
Epoch: 90 | Test Accuracy: 0.9388888888888889


In [87]:
tanh_train = train(
    X_train, y_train_encoded, 
    X_test, y_test, y_test_encoded, 
    activation='tanh',
    epoch = 100,
    learning_rate=0.5
)

Epoch: 0 | Test Accuracy: 0.41388888888888886
Epoch: 10 | Test Accuracy: 0.8083333333333333
Epoch: 20 | Test Accuracy: 0.8722222222222222
Epoch: 30 | Test Accuracy: 0.8833333333333333
Epoch: 40 | Test Accuracy: 0.8972222222222223
Epoch: 50 | Test Accuracy: 0.9166666666666666
Epoch: 60 | Test Accuracy: 0.9166666666666666
Epoch: 70 | Test Accuracy: 0.9305555555555556
Epoch: 80 | Test Accuracy: 0.9361111111111111
Epoch: 90 | Test Accuracy: 0.9361111111111111


Here, used hyperparameters are, epoch = 100 and learning_rate = 0.5. With these hyperparameters, the accuracies with different activation functions are below:
sigmoid = 0.930, relu = 0.938, and tanh = 0.936
There results for all the cases are quite similar.